# Transformer Foundations, Part 5: Build the Pretraining Data Stream

## Start With the Job: Decide What the Model Will Practice

The model from Part 4 accepts token IDs. Raw documents are not token IDs, and a folder of text is not yet a trustworthy dataset. This chapter builds the missing left side of the learning spine:

```mermaid
flowchart LR
    A["Raw documents"] --> B["Inventory + hashes"]
    B --> C["Whole-document train/validation split"]
    C --> D["Duplicate + source-mixture audit"]
    D --> E["Fit tokenizer on training text only"]
    E --> F["Insert document boundaries"]
    F --> G["Pack fixed-length token blocks"]
    G --> H["Verified shards + manifest"]
    H --> I["Token IDs enter the decoder"]
```

Mental model: **the pipeline writes the model's practice book**. Repeated passages receive extra votes. A leaked validation chapter turns the exam into homework. Missing document boundaries teach false transitions. Padding spends compute practicing emptiness.

## 0. The Challenge and Scope

> **The mission:** Turn the repository's committed text corpus into a reproducible, leakage-checked next-token stream without changing any source document.

**What we know so far:** the modern decoder has a vocabulary size and context length waiting for data. **But a folder is not a dataset:** it has no frozen split, tokenizer identity, boundary token, packing rule, shard hash, or reload proof.

**What this chapter unlocks:** document ownership before fitting, transparent contamination diagnostics, a training-only byte-level BPE tokenizer, fixed-length shards, and a manifest that can refuse altered bytes.

| Topic | Coverage | Intuitive reason |
|---|---|---|
| Inventory, whole-document split, duplicate checks | Built and measured | Protect the meaning of held-out validation |
| Source mixture | Built and measured | Make the practice budget explicit |
| Byte-level BPE tokenizer | Built and audited | Freeze the model's alphabet and word-part dictionary |
| EOD and packing | Built and measured | Preserve boundaries while filling useful context positions |
| Shards, hashes, and decoded reload | Built and measured | Prove the bytes on disk match the intended dataset |
| Web-scale filtering and distributed sharding | Named only | These need policy and infrastructure beyond this local build |

### Roadmap

| Part | Failure first | Evidence |
|---|---|---|
| 1 | Unknown corpus shape | Document, byte, token, and exclusion counts |
| 2 | Adjacent prose leaks | Disjoint whole-document ownership |
| 3 | Copies hide behind names | Exact hashes and near-duplicate similarity |
| 4 | Accidental practice budgets | Source-mixture bars |
| 5 | No stable model alphabet | Training-only tokenizer fit audit |
| 6 | Document edges vanish | Atomic end-of-document token |
| 7 | Padding wastes blocks | Measured useful-token density |
| 8 | Successful writes are mistaken for trusted writes | Hashes and decoded reloads |

**Predict:** Which mistake cannot be repaired after tokenizer fitting: a poor plot color, a leaked validation chapter, or a short tail block? The fit audit resolves it.

In [ ]:
# -- Reproducible setup and immutable corpus inventory ----------------------
from __future__ import annotations
import hashlib, html, json, os, random, re
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
try:
    from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
except ImportError as exc:
    raise ImportError("Install `tokenizers` in the genai-01-transformers kernel with `pip install tokenizers`, restart, and rerun.") from exc
SEED, CONTEXT_LENGTH, TARGET_VOCAB_SIZE = 2505, 128, 4096
EOD_TOKEN, PAD_TOKEN, UNK_TOKEN, BOS_TOKEN = "<|endoftext|>", "<|pad|>", "<|unk|>", "<|bos|>"
DARK, BLUE, AMBER, GREEN, RED = "#1a1a2e", "#60a5fa", "#f59e0b", "#22c55e", "#ef4444"
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "figure.facecolor": DARK, "axes.facecolor": DARK})
def find_repo_root():
    start = Path(os.environ.get("AI_PORTFOLIO_ROOT", Path.cwd())).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "AUTHORING_GUIDE.md").is_file() and (candidate / "learning/genai/content").is_dir(): return candidate
    raise FileNotFoundError("Set AI_PORTFOLIO_ROOT to the ai-portfolio repository root.")
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
def relative(path): return path.relative_to(REPO_ROOT).as_posix()
REPO_ROOT = find_repo_root(); CONTENT_ROOT = REPO_ROOT / "learning/genai/content"
CHAPTER_ROOT = REPO_ROOT / "learning/genai/01-transformers"
ARTIFACT_ROOT = Path(os.environ.get("RIVERSIDE_ARTIFACT_ROOT", CHAPTER_ROOT / "artifacts/base-lm")).resolve()
all_corpus_files = sorted(p for p in CONTENT_ROOT.rglob("*") if p.is_file())
source_snapshot_before = {relative(p): sha256_file(p) for p in all_corpus_files}
document_paths = sorted(CONTENT_ROOT.glob("*/chapter-*.txt")); excluded_paths = sorted(set(all_corpus_files) - set(document_paths))
records = []
for path in document_paths:
    payload = path.read_bytes()
    records.append({"path": relative(path), "source": path.parent.name, "bytes": len(payload), "sha256": hashlib.sha256(payload).hexdigest(), "text": payload.decode("utf-8")})
assert records and len({r["path"] for r in records}) == len(records)
sources = sorted({r["source"] for r in records})
document_counts = [sum(r["source"] == source for r in records) for source in sources]
byte_counts = [sum(r["bytes"] for r in records if r["source"] == source) for source in sources]
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=DARK)
for axis, values, title, color in zip(axes, [document_counts, byte_counts], ["documents", "UTF-8 bytes"], [BLUE, AMBER]):
    axis.barh(sources, values, color=color); axis.set_title(title, color="white"); axis.tick_params(colors="white")
fig.suptitle("Pre-tokenizer inventory: source budgets already differ", color="white"); fig.tight_layout(); plt.show()
print(f"Repository: {REPO_ROOT}\nArtifact target: {ARTIFACT_ROOT}")
print(f"Chapters={len(records):,}; bytes={sum(r['bytes'] for r in records):,}; exclusions={len(excluded_paths):,}")
print("PASS: every committed corpus file has an immutable pre-run digest")

**Code Walkthrough: setup and inventory**

The import failure names the recovery command. Root discovery requires both the authoring guide and corpus. `RIVERSIDE_ARTIFACT_ROOT` supports temporary validation. Candidates are read as bytes before UTF-8 decoding, only `chapter-*.txt` enters training, and every other corpus file remains visible in the exclusion list.

## 1. Inventory, Then Split by Document

A random token split tears neighboring paragraphs from one chapter into both sets. Whole-document ownership is the smallest useful boundary here.

```mermaid
flowchart TB
 A["One continuous chapter"] --> B["Correct: one owner"]
 A --> C["Wrong: shuffled fragments"]
 C --> D["Training fragment"]
 C --> E["Validation fragment"]
 D -. "adjacent prose leaks" .-> E
 B --> F["Different validation chapters"]
 style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** One global shuffle or a seeded per-source split: which keeps every novel represented on both sides?

In [ ]:
# -- Make and visualize a deterministic document-level split ---------------
def split_documents(items, seed, validation_fraction=0.12):
    by_source = defaultdict(list)
    for item in items: by_source[item["source"]].append(item)
    train_items, validation_items = [], []
    for source in sorted(by_source):
        group = sorted(by_source[source], key=lambda item: item["path"])
        local_rng = random.Random(f"{seed}:{source}"); local_rng.shuffle(group)
        count = min(max(1, round(len(group) * validation_fraction)), len(group) - 1)
        validation_items.extend(group[:count]); train_items.extend(group[count:])
    return sorted(train_items, key=lambda item: item["path"]), sorted(validation_items, key=lambda item: item["path"])
train_records, validation_records = split_documents(records, SEED)
train_paths = {r["path"] for r in train_records}; validation_paths = {r["path"] for r in validation_records}
assert train_paths.isdisjoint(validation_paths)
assert train_paths | validation_paths == {r["path"] for r in records}
assert {r["source"] for r in train_records} == {r["source"] for r in validation_records}
fig, axes = plt.subplots(1, 2, figsize=(13, 4), facecolor=DARK)
for axis in axes: axis.axis("off")
for index in range(8):
    axes[0].add_patch(plt.Rectangle((index, .35), .75, .35, color=BLUE if index < 6 else GREEN))
    axes[0].text(index + .37, .52, f"D{index+1}", ha="center", va="center", color="white")
for index in range(16): axes[1].add_patch(plt.Rectangle((index/2, .35), .38, .35, color=RED if index % 4 == 0 else BLUE))
for axis, title in zip(axes, ["Correct: whole-document ownership", "Wrong: adjacent fragments leak"]):
    axis.set_xlim(-.2, 8); axis.set_ylim(0, 1); axis.set_title(title, color="white")
plt.show()
print(f"Training documents={len(train_records)}; validation documents={len(validation_records)}")
print("PASS: paths are disjoint, exhaustive, deterministic, and source-stratified")
print("Prediction resolved: per-source splitting keeps every Riverside source on both sides.")

## 2. Exact and Near-Duplicate Diagnostics

Different filenames do not imply different text. Exact hashes catch normalized copies; transparent five-word shingle overlap catches light edits. Diagnostic copies stay in memory and never enter either split.

```mermaid
flowchart LR
 A["Original passage"] --> B["Exact copy"] --> D["Same hash"]
 A --> C["Light edit"] --> E["High overlap"]
 D --> F["Detector fires"]
 E --> F
 style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Warning:** Similar topics are not automatically leakage. Natural findings are reported, never silently removed.

In [ ]:
# -- Detect injected and natural duplicates, then animate thresholding -------
def normalize_text(text): return re.sub(r"\s+", " ", text.casefold()).strip()
def shingles(text, width=5, cap=240):
    words = re.findall(r"[a-z0-9']+", normalize_text(text))
    values = [" ".join(words[i:i+width]) for i in range(max(0, len(words)-width+1))]
    if len(values) <= cap: return set(values)
    # Hash-ranked sampling stays stable when a small edit shifts later shingle positions.
    return set(sorted(values, key=lambda value: hashlib.sha256(value.encode()).digest())[:cap])
def jaccard(left, right):
    union = left | right
    return len(left & right) / len(union) if union else 1.0
base = records[0]["text"][:5000]
diagnostic_texts = [base, base, base.replace(" the ", " a ", 4) + "\nA small diagnostic edit."]
labels = ["original", "exact copy", "light edit"]
diagnostic_hashes = [hashlib.sha256(normalize_text(text).encode()).hexdigest() for text in diagnostic_texts]
diagnostic_shingles = [shingles(text) for text in diagnostic_texts]
similarity = np.array([[jaccard(left, right) for right in diagnostic_shingles] for left in diagnostic_shingles])
NEAR_DUPLICATE_THRESHOLD = 0.85
assert diagnostic_hashes[0] == diagnostic_hashes[1] and similarity[0, 2] >= NEAR_DUPLICATE_THRESHOLD
groups = defaultdict(list)
for record in records: groups[hashlib.sha256(normalize_text(record["text"]).encode()).hexdigest()].append(record["path"])
natural_exact_groups = [sorted(paths) for paths in groups.values() if len(paths) > 1]
record_shingles = {record["path"]: shingles(record["text"]) for record in records}
natural_near_groups = []
for left_index, left in enumerate(records):
    for right in records[left_index+1:]:
        if left["sha256"] == right["sha256"]: continue
        score = jaccard(record_shingles[left["path"]], record_shingles[right["path"]])
        if score >= NEAR_DUPLICATE_THRESHOLD:
            natural_near_groups.append({"documents": [left["path"], right["path"]], "similarity": round(score, 4)})
fig, axis = plt.subplots(figsize=(6, 5), facecolor=DARK); image = axis.imshow(similarity, vmin=0, vmax=1, cmap="viridis")
axis.set_xticks(range(3), labels, rotation=25, ha="right", color="white"); axis.set_yticks(range(3), labels, color="white")
for row in range(3):
    for column in range(3): axis.text(column, row, f"{similarity[row, column]:.2f}", ha="center", va="center", color="white")
fig.colorbar(image, ax=axis); axis.set_title("Injected similarity matrix", color="white"); plt.show()
thresholds = np.linspace(.55, 1.0, 10); positions = np.array([[.1, .2], [.9, .2], [.5, .85]])
fig, axis = plt.subplots(figsize=(7, 5), facecolor=DARK)
def draw_threshold(frame):
    axis.clear(); axis.set_xlim(0, 1); axis.set_ylim(0, 1); axis.axis("off"); threshold = thresholds[frame]
    for left in range(3):
        for right in range(left+1, 3):
            color = GREEN if similarity[left, right] >= threshold else "#475569"
            axis.plot(*zip(positions[left], positions[right]), color=color, linewidth=3 if color == GREEN else 1)
    axis.scatter(positions[:, 0], positions[:, 1], s=1500, color=[BLUE, AMBER, GREEN])
    for index, label in enumerate(labels): axis.text(*positions[index], label, ha="center", va="center", color="white")
    axis.set_title(f"Pairs retained at threshold {threshold:.2f}", color="white")
animation = FuncAnimation(fig, draw_threshold, frames=len(thresholds), interval=450, repeat=True)
animation_html = animation.to_jshtml(); plt.close(fig)
print("Watch the light edit disconnect as the threshold tightens; the exact copy persists."); display(HTML(animation_html))
print(f"Natural exact groups={len(natural_exact_groups)}; natural near pairs={len(natural_near_groups)}")
print("PASS: injected exact and lightly edited copies were detected in memory")

**Code Walkthrough: duplicate evidence and animation**

Normalization changes only case and repeated whitespace. Sampled five-word shingles cap CPU work while preserving phrase identity. Every animation edge comes from the measured matrix; frames change the threshold, not the evidence. Natural pairs remain in manifest accounting.

## 3. Source Mixture and Training-Only Byte-Level BPE

Equal-document sampling gives each chapter one vote. Proportional-token sampling gives each token one vote. Neither policy is universal; accidental policy is the failure. Byte-level BPE makes the actual token budget measurable, but validation must never enter fitting.

```mermaid
flowchart LR
 A["Training paths"] --> B["Audited iterator"] --> C["Byte-level BPE"] --> D["Frozen IDs"]
 E["Validation paths"] --> F["Blocked from fit"]
 D --> G["Encode both splits"]
 F --> G
 style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** If a validation-only name fragments more, is that leakage or honest held-out behavior?

**Warning:** The library exposes no reliable intermediate merge snapshots. This notebook uses measured static before/after evidence instead of a fabricated vocabulary-growth animation.

In [ ]:
# -- Fit BPE on training only and measure inventory, mixture, fragmentation --
fit_audit_paths, fit_audit_hashes = [], []
def training_iterator():
    for record in train_records:
        fit_audit_paths.append(record["path"]); fit_audit_hashes.append(record["sha256"]); yield record["text"]
tokenizer = Tokenizer(models.BPE(unk_token=UNK_TOKEN))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False); tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=TARGET_VOCAB_SIZE, min_frequency=2, show_progress=False, special_tokens=[PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOD_TOKEN])
tokenizer.train_from_iterator(training_iterator(), trainer=trainer, length=len(train_records))
vocab_size = tokenizer.get_vocab_size(); validation_hashes = {record["sha256"] for record in validation_records}
assert set(fit_audit_paths) == train_paths and set(fit_audit_paths).isdisjoint(validation_paths)
assert set(fit_audit_hashes).isdisjoint(validation_hashes), "Exact validation payload leaked through a training duplicate"
assert all(tokenizer.token_to_id(token) is not None for token in [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOD_TOKEN])
assert vocab_size <= np.iinfo(np.uint16).max + 1
for record in records: record["token_count"] = len(tokenizer.encode(record["text"]).ids) + 1
token_counts = [sum(r["token_count"] for r in records if r["source"] == source) for source in sources]
equal_share = np.array(document_counts) / sum(document_counts); token_share = np.array(token_counts) / sum(token_counts)
fig, axes = plt.subplots(1, 3, figsize=(17, 5), facecolor=DARK)
for axis, values, title, color in zip(axes, [document_counts, byte_counts, token_counts], ["documents", "bytes", "frozen BPE tokens"], [BLUE, AMBER, GREEN]):
    axis.barh(sources, values, color=color); axis.set_title(title, color="white"); axis.tick_params(colors="white")
fig.suptitle("Inventory by source: all values measured", color="white"); fig.tight_layout(); plt.show()
fig, axis = plt.subplots(figsize=(12, 5), facecolor=DARK); x = np.arange(len(sources)); width = .38
axis.bar(x-width/2, equal_share, width, label="equal document", color=BLUE); axis.bar(x+width/2, token_share, width, label="proportional token", color=AMBER)
axis.set_xticks(x, sources, rotation=35, ha="right"); axis.tick_params(colors="white"); axis.legend(); axis.set_title("Source-mixture allocation", color="white"); plt.show()
samples = ["Aria listened aboard Meridian.", "The Weight of Distant Light -- chapter twelve.", "Ordinary prose remains readable, even with punctuation."]
gpt2 = None
try:
    from transformers import GPT2TokenizerFast
    gpt2 = GPT2TokenizerFast.from_pretrained("gpt2", local_files_only=True)
except Exception as exc:
    print(f"GPT-2 comparison unavailable offline: {type(exc).__name__}; cache the real tokenizer files to enable it.")
for sample in samples:
    ours = tokenizer.encode(sample)
    print(f"Text: {sample}\n  Riverside: {ours.tokens}\n  GPT-2: {gpt2.tokenize(sample) if gpt2 else '[real tokenizer not cached]'}")
    assert tokenizer.decode(ours.ids) == sample
fragment_labels = ["Aria", "Meridian", "Distant Light", "ordinary prose", "-- punctuation --"]
pieces = [tokenizer.encode(label).tokens for label in fragment_labels]
fig, axis = plt.subplots(figsize=(12, 5), facecolor=DARK); axis.set_xlim(0, max(map(len, pieces))+1); axis.set_ylim(-.8, len(pieces)-.2)
axis.set_yticks(range(len(pieces)), fragment_labels, color="white"); axis.tick_params(colors="white")
for row, parts in enumerate(pieces):
    for column, piece in enumerate(parts):
        axis.add_patch(plt.Rectangle((column, row-.3), .9, .6, color=[BLUE, AMBER, GREEN][column%3])); axis.text(column+.45, row, piece.replace("Ġ", "_"), ha="center", va="center", color="white", fontsize=8)
axis.set_title("Measured fragmentation: every box is a real token", color="white"); plt.show()
print(f"Vocabulary={vocab_size:,}; largest mixture shift={np.max(np.abs(equal_share-token_share)):.1%}")
print("PASS: validation paths and exact validation payloads never entered fitting")
print("Prediction resolved: held-out fragmentation is honest boundary evidence, not a reason to refit.")

**Code Walkthrough: audited tokenizer and real comparisons**

The iterator records every path and payload digest immediately before yielding text. Assertions require exact training-set equality and validation disjointness. Inventory then gains real BPE token counts. GPT-2 pieces appear only from a cached real tokenizer; Riverside round trips are always checked. The fragmentation view uses actual token pieces, not merge folklore.

## 4. Insert Explicit Document Boundaries

Concatenation without a marker teaches a false transition from one chapter ending to the next opening. EOD makes the boundary an explicit prediction.

```mermaid
flowchart LR
 A["Chapter A ending"] --> B["No marker"] --> C["Chapter B opening"]
 A --> D["EOD"] --> E["Predict boundary"] --> C
 style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Your turn:** Change `preview_chars` below and verify EOD remains one token.

In [ ]:
# -- Prove EOD is explicit and atomic --------------------------------------
preview_chars = 90  # CHANGE THIS: inspect a wider or narrower boundary.
left = train_records[0]["text"][-preview_chars:]; right = train_records[1]["text"][:preview_chars]
eod_id = tokenizer.token_to_id(EOD_TOKEN)
without_eod = tokenizer.encode(left).ids + tokenizer.encode(right).ids
with_eod = tokenizer.encode(left).ids + [eod_id] + tokenizer.encode(right).ids
assert tokenizer.encode(EOD_TOKEN).ids == [eod_id]
print("WITHOUT EOD:\n" + tokenizer.decode(without_eod, skip_special_tokens=False))
print("\nWITH EOD:\n" + tokenizer.decode(with_eod, skip_special_tokens=False).replace(EOD_TOKEN, "[EOD]"))
print(f"PASS: EOD is one token with ID {eod_id}; the edge is now a target.")

## 5. Pack Independent Fixed-Length Streams

One-document-per-block padding preserves boundaries but wastes width. Stream packing keeps EOD, fills complete blocks, and leaves one tail per split. Training and validation use separate calls.

```mermaid
flowchart TB
 A["Training docs plus EOD"] --> B["Training stream"] --> C["Full blocks"]
 D["Validation docs plus EOD"] --> E["Validation stream"] --> F["Full blocks"]
 B --> G["Measured train tail"]
 E --> H["Measured validation tail"]
 style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style D fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Which wastes more: one tail per document or one tail per split?

In [ ]:
# -- Encode, pack, animate real tokens, and compare utilization -------------
def encode_documents(items):
    encoded = []
    for item in items:
        ids = tokenizer.encode(item["text"]).ids
        assert ids, f"No tokens: {item['path']}"
        encoded.append({"path": item["path"], "source": item["source"], "ids": ids + [eod_id]})
    return encoded
def pack_documents(encoded):
    stream = np.asarray([token_id for item in encoded for token_id in item["ids"]], dtype=np.int64)
    written = (len(stream) // CONTEXT_LENGTH) * CONTEXT_LENGTH
    blocks, tail = stream[:written].reshape(-1, CONTEXT_LENGTH), stream[written:]
    return blocks, tail, {"input_tokens": len(stream), "written_tokens": written, "tail_tokens": len(tail), "blocks": len(blocks), "eod_in_written_blocks": int(np.count_nonzero(blocks == eod_id))}
encoded_train, encoded_validation = encode_documents(train_records), encode_documents(validation_records)
train_blocks, train_tail, train_pack_stats = pack_documents(encoded_train)
validation_blocks, validation_tail, validation_pack_stats = pack_documents(encoded_validation)
assert train_blocks.size and validation_blocks.size and max(train_blocks.max(), validation_blocks.max()) < vocab_size
assert all(len(item["ids"]) > 1 for item in encoded_train + encoded_validation)
lengths = [len(item["ids"]) for item in encoded_train]; useful = sum(lengths)
padded_capacity = sum(int(np.ceil(length/CONTEXT_LENGTH))*CONTEXT_LENGTH for length in lengths)
packed_capacity = int(np.ceil(useful/CONTEXT_LENGTH))*CONTEXT_LENGTH
padded_util, packed_util = useful/padded_capacity, useful/packed_capacity
fig, axis = plt.subplots(figsize=(8, 4), facecolor=DARK)
bars = axis.bar(["one tail per document", "one tail per split"], [padded_util, packed_util], color=[RED, GREEN]); axis.set_ylim(0, 1.05); axis.tick_params(colors="white")
for bar, value in zip(bars, [padded_util, packed_util]): axis.text(bar.get_x()+bar.get_width()/2, value+.02, f"{value:.1%}", ha="center", color="white")
axis.set_title("Measured capacity utilization", color="white"); plt.show()
preview_ids = [token_id for item in encoded_train[:3] for token_id in item["ids"]]; width = 24
frames = np.linspace(1, min(len(preview_ids), width*4), 14, dtype=int)
fig, axis = plt.subplots(figsize=(12, 4), facecolor=DARK)
def draw_packing(frame):
    axis.clear(); axis.axis("off"); filled = frames[frame]
    for position in range(width*4):
        row, column = divmod(position, width)
        color = "#334155" if position >= filled else (GREEN if preview_ids[position] == eod_id else (BLUE if row % 2 == 0 else AMBER))
        axis.add_patch(plt.Rectangle((column, -row), .88, .72, color=color))
    axis.set_xlim(0, width); axis.set_ylim(-3.8, 1); axis.set_title(f"Real stream fill: {filled} tokens; green is EOD", color="white")
packing_animation = FuncAnimation(fig, draw_packing, frames=len(frames), interval=400, repeat=True)
packing_html = packing_animation.to_jshtml(); plt.close(fig)
print("Watch real tokens cross block rows while green EOD markers preserve boundaries."); display(HTML(packing_html))
print(f"Training={train_pack_stats}\nValidation={validation_pack_stats}")
print("PASS: every document contributed tokens; splits were packed independently.")
print("Prediction resolved: one tail per split wastes less measured capacity.")

**Code Walkthrough: fixed-length packing**

Every document gets EOD before flattening. One call owns one split, writes complete blocks, and returns its tail visibly. Range checks precede the cast. Utilization comes from actual encoded lengths, and the animation colors actual EOD IDs rather than scripted boundaries.

## 6. Write, Reload, Hash, and Decode

A successful write is not proof. The boundary closes only after the tokenizer reloads, shard hashes match disk bytes, IDs fit vocabulary and dtype, and three windows show human-readable prose plus boundaries.

```mermaid
flowchart LR
 A["Source paths and hashes"] --> B["Frozen split"] --> C["Fit audit"]
 C --> D["tokenizer.json"]
 C --> E["train.bin"]
 C --> F["validation.bin"]
 D --> G["dataset-manifest.json"]
 E --> G
 F --> G
 G --> H["Notebook 06 digest refusal"]
 style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Warning:** `uint16` is valid only because the measured maximum ID fits. Check before casting and after reload.

In [ ]:
# -- Write frozen artifacts, reload them, and render decoded windows --------
TOKENIZER_ROOT = ARTIFACT_ROOT / "tokenizer"; TOKENIZER_ROOT.mkdir(parents=True, exist_ok=True)
TOKENIZER_PATH = TOKENIZER_ROOT / "tokenizer.json"; TRAIN_PATH = ARTIFACT_ROOT / "train.bin"
VALIDATION_PATH = ARTIFACT_ROOT / "validation.bin"; MANIFEST_PATH = ARTIFACT_ROOT / "dataset-manifest.json"
tokenizer.save(str(TOKENIZER_PATH))
train_uint16 = train_blocks.astype(np.uint16, copy=False).reshape(-1)
validation_uint16 = validation_blocks.astype(np.uint16, copy=False).reshape(-1)
assert int(train_uint16.max()) < vocab_size and int(validation_uint16.max()) < vocab_size
train_uint16.tofile(TRAIN_PATH); validation_uint16.tofile(VALIDATION_PATH)
train_hash, validation_hash = sha256_file(TRAIN_PATH), sha256_file(VALIDATION_PATH); assert train_hash != validation_hash
source_contributions = {source: int(sum(r["token_count"] for r in records if r["source"] == source)) for source in sources}
assert all(source_contributions.values())
manifest = {
    "schema_version": 1, "seed": SEED, "tokenizer_type": "byte_level_bpe", "vocab_size": vocab_size,
    "eod_token": EOD_TOKEN, "context_length": CONTEXT_LENGTH, "dtype": "uint16",
    "source_documents": [r["path"] for r in records], "train_documents": [r["path"] for r in train_records],
    "validation_documents": [r["path"] for r in validation_records], "excluded_documents": [relative(path) for path in excluded_paths],
    "exact_duplicate_groups": natural_exact_groups, "near_duplicate_groups": natural_near_groups,
    "train_tokens": int(train_uint16.size), "validation_tokens": int(validation_uint16.size),
    "train_sha256": train_hash, "validation_sha256": validation_hash, "tokenizer_sha256": sha256_file(TOKENIZER_PATH),
    "tokenizer_fit_documents": fit_audit_paths, "tokenizer_fit_payload_sha256": fit_audit_hashes,
    "special_tokens": {"pad": PAD_TOKEN, "unk": UNK_TOKEN, "bos": BOS_TOKEN, "eos_eod": EOD_TOKEN},
    "near_duplicate_threshold": NEAR_DUPLICATE_THRESHOLD,
    "injected_duplicate_diagnostics": {"exact_detected": True, "near_detected": True, "near_similarity": round(float(similarity[0, 2]), 4)},
    "source_token_contributions": source_contributions,
    "packing": {"train": train_pack_stats, "validation": validation_pack_stats},
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
reloaded_manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
reloaded_tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
reloaded_train = np.fromfile(TRAIN_PATH, dtype=np.uint16); reloaded_validation = np.fromfile(VALIDATION_PATH, dtype=np.uint16)
assert sha256_file(TRAIN_PATH) == reloaded_manifest["train_sha256"]
assert sha256_file(VALIDATION_PATH) == reloaded_manifest["validation_sha256"]
assert reloaded_train.size == reloaded_manifest["train_tokens"] and reloaded_validation.size == reloaded_manifest["validation_tokens"]
assert max(int(reloaded_train.max()), int(reloaded_validation.max())) < vocab_size
assert reloaded_tokenizer.get_vocab_size() == vocab_size
rng = np.random.default_rng(SEED); window_size = 72
starts = np.arange(0, reloaded_train.size-window_size, CONTEXT_LENGTH)
window_starts = sorted(rng.choice(starts, size=3, replace=False).tolist()); decoded_windows = []
for start in window_starts:
    decoded = reloaded_tokenizer.decode(reloaded_train[start:start+window_size].astype(int).tolist(), skip_special_tokens=False).replace(EOD_TOKEN, "[EOD]").replace(PAD_TOKEN, "[PAD]")
    decoded_windows.append((start, decoded)); print(f"Window at token {start:,}: {decoded[:320]!r}")
fig, axes = plt.subplots(3, 1, figsize=(14, 8), facecolor=DARK)
for axis, (start, decoded) in zip(axes, decoded_windows):
    axis.axis("off"); wrapped = "\n".join(decoded[index:index+120] for index in range(0, min(len(decoded), 480), 120))
    axis.text(.01, .92, f"token {start:,}", color=GREEN, va="top")
    axis.text(.01, .72, wrapped, color="white", fontsize=9, va="top", family="monospace")
fig.suptitle("Three deterministic decoded windows; [EOD] marks boundaries", color="white"); fig.tight_layout(); plt.show()
rows = [("documents", len(records)), ("train / validation docs", f"{len(train_records)} / {len(validation_records)}"), ("vocabulary", vocab_size), ("context", CONTEXT_LENGTH), ("train / validation tokens", f"{reloaded_train.size:,} / {reloaded_validation.size:,}"), ("exact / near findings", f"{len(natural_exact_groups)} / {len(natural_near_groups)}"), ("dtype", "uint16")]
table = "<table><tr><th>Manifest field</th><th>Measured value</th></tr>" + "".join(f"<tr><td>{html.escape(str(key))}</td><td>{html.escape(str(value))}</td></tr>" for key, value in rows) + "</table>"
display(HTML(table))
print("PASS: frozen contract written; hashes, ranges, tokenizer, and three decoded windows reloaded cleanly.")

**Code Walkthrough: writer, lineage, and clean reload**

Range checks precede the cast. Shards are hashed from disk and those digests enter the frozen manifest fields expected by Notebook 06. Extra fields preserve fit evidence, duplicate accounting, source contributions, special tokens, and tails. Reload uses only disk files; three block-aligned windows expose real prose and EOD boundaries.

In [ ]:
# -- Run final source, leakage, duplicate, range, and hash checks ------------
source_snapshot_after = {relative(path): sha256_file(path) for path in all_corpus_files}
assert source_snapshot_after == source_snapshot_before, "A committed corpus file changed"
assert set(manifest["train_documents"]).isdisjoint(manifest["validation_documents"])
assert set(manifest["tokenizer_fit_documents"]) == set(manifest["train_documents"])
assert set(manifest["tokenizer_fit_documents"]).isdisjoint(manifest["validation_documents"])
assert manifest["injected_duplicate_diagnostics"]["exact_detected"]
assert manifest["injected_duplicate_diagnostics"]["near_detected"]
assert all(value > 0 for value in manifest["source_token_contributions"].values())
assert reloaded_train.dtype == np.uint16 and reloaded_validation.dtype == np.uint16
assert len(decoded_windows) == 3
print("PASS: source files are byte-for-byte unchanged")
print("PASS: document paths are disjoint and validation never entered tokenizer fitting")
print("PASS: duplicate injections, natural accounting, token range, uint16 fit, source contribution, hash re-read, and three decodes passed")

## 7. Scorecard, Reflections, and Three-Tier Ledger

```mermaid
flowchart LR
 A["Raw files"] --> B["Frozen ownership"] --> C["Audited tokenizer"] --> D["Boundary-aware shards"] --> E["Reloaded identities"]
 style A fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Constraint | Before | After |
|---|---|---|
| source identity | folder names | every corpus file hashed before and after |
| validation ownership | undefined | deterministic disjoint document paths |
| contamination evidence | none | injected checks plus natural accounting |
| mixture | accidental | equal-document and proportional-token budgets |
| vocabulary | undefined | training-only byte-level BPE and special tokens |
| boundaries | invisible | one atomic EOD after every document |
| batches | padding-prone | independent full blocks with measured tails |
| artifacts | untrusted writes | manifest, disk hashes, clean reload, three decodes |

**Checkpoint:** Riverside now has a reproducible data contract, not a claim that this small corpus is sufficient for a capable model.

**Reflection:** The decisive operation happened before tokenization: assigning whole documents to one side. Later hashes prove identity, but none can undo leaked ownership.

**Your turn:** Change `CONTEXT_LENGTH` to 64, rerun into a temporary `RIVERSIDE_ARTIFACT_ROOT`, and compare tails and utilization. Source digests must remain unchanged.

| Tier | Techniques | Evidence or reason |
|---|---|---|
| Built and measured | inventory, immutable hashes, source-stratified split, exact hashes, shingle similarity, mixture, byte-level BPE, EOD, packing, `uint16` shards, manifest, reload | executable checks and measured visuals |
| Explained and illustrated | token-level leakage, equal-document sampling, proportional-token sampling, per-document padding | diagrams and actual comparisons |
| Named with a reason | language ID, PII removal, license policy, quality classifiers, semantic deduplication, distributed sharding | requires governance, learned models, or infrastructure |

If you find a technique named above that does not appear in the tier table, that is exactly the bug this section exists to catch.

### Key Takeaways

- Split documents before text-dependent fitting.
- Record duplicates; never make them disappear silently.
- Treat source mixture as a declared budget.
- Insert boundaries before packing.
- Trust reloaded hashes, not successful write calls.

> **Next:** `06-pretrain-a-base-model.ipynb` reloads this tokenizer and both shards from a fresh kernel, refuses digest mismatches, and trains from random weights.